# Evans Index Z (EI-z) — MRI Adaptation

Measures the maximum height of the frontal horns on coronal slices from the AC to 20mm below, normalized by brain height at the same slice.

**MRI adaptation**: No HU windowing. For OASIS (atlas-registered isotropic), coronal slices are extracted via direct numpy indexing `arr[:, y, :]`. For ABCD, use `extract_coronal_view()`.

**Reference**: Samadani et al., JNS 2024

## Library imports

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
from skimage import measure, filters
import matplotlib.pyplot as plt

from mri_utils import (rotate_image, rotate_image_coronal, getLargestCC, extract_coronal_view,
                       normalize_to_uint8,
                       OASIS_AC_VOXEL, OASIS_PC_VOXEL, parse_acpc_coords, load_acpc_from_txt)

## Configuration

Set `DATASET` to match your data source. The notebook adapts file discovery and AC coordinate loading accordingly.

Supported datasets:
- **`"oasis1"`** — OASIS-1 cross-sectional (skull-stripped, atlas-registered `.img` files). AC/PC are fixed atlas voxel coordinates from `mri_utils.OASIS_AC_VOXEL`.
- **`"abcd"`** — ABCD Study T1w scans (ART-corrected `.nii`/`.nii.gz` files). AC/PC loaded per-subject from `.txt` sidecars.
- **`"custom"`** — Any NIfTI dataset; set `SCAN_GLOB` and provide AC coordinates.

In [ ]:
# ── Choose your dataset ──────────────────────────────────────────────
DATASET = "oasis1"  # "oasis1", "abcd", or "custom"

# ── Dataset-specific settings ────────────────────────────────────────
if DATASET == "oasis1":
    DATA_DIR = "data/oasis1/disc1"
    DEMOGRAPHICS_PATH = "data/oasis1/oasis_cross-sectional-5708aa0a98d82080.xlsx"
    # OASIS skull-stripped, gain-field corrected, atlas-registered volumes
    SCAN_GLOB = "OAS1_*/PROCESSED/MPRAGE/T88_111/*_masked_gfc.img"
    SUBJECT_ID_FROM_PATH = lambda p: p.split(os.sep)[-5]  # e.g. OAS1_0001_MR1
    # For OASIS atlas-registered isotropic volumes, AC is a fixed atlas voxel
    # ac_indices = (x_vox, y_vox, z_vox); coronal slices are arr[:, y_vox, :]
    AC_MODE = "oasis_atlas"   # use OASIS_AC_VOXEL from mri_utils
    ACPC_TXT_GLOB = None

elif DATASET == "abcd":
    DATA_DIR = "data/abcd"
    DEMOGRAPHICS_PATH = None  # set path to ABCD demographics CSV/TSV if available
    # ART-corrected T1w scans (output from Charlie's ART batch script)
    SCAN_GLOB = "**/*_T1w_acpc.nii*"
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).split("_ses")[0]  # e.g. sub-NDARINVXXXXXXXX
    # AC/PC are per-subject; load from sidecar .txt files
    AC_MODE = "txt_sidecar"   # load_acpc_from_txt() per subject
    ACPC_TXT_GLOB = "**/*_acpc_coords.txt"  # adjust to your naming

elif DATASET == "custom":
    DATA_DIR = "data/my_dataset"
    DEMOGRAPHICS_PATH = None
    SCAN_GLOB = "**/*.nii.gz"  # adjust to your file naming
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")
    AC_MODE = "oasis_atlas"   # or "txt_sidecar"
    ACPC_TXT_GLOB = None

else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Use 'oasis1', 'abcd', or 'custom'.")

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset:      {DATASET}")
print(f"Data dir:     {DATA_DIR}")
print(f"Scan pattern: {SCAN_GLOB}")
print(f"AC mode:      {AC_MODE}")

## Processing parameters

In [ ]:
coronal_seg_params = {
    "gauss_blur_kernel_size": 3,
    "adaptive_thresh_size": 75,
    "adaptive_thresh_C": 7.5,
    "upper_percentile": 20,
    "lower_percentile": 80,
    "close_kernel_size": 5,
    "close_iterations": 2,
    "open_kernel_size": 3,
    "open_iterations": 2,
    "num_slices_below_ac": 20,
}

## Coronal segmentation functions

Three functions implement the coronal ventricle segmentation pipeline for MRI:

1. **`segment_coronal_plane`** — Core 2D segmentation. Adapts the CT pipeline by replacing HU windowing with Otsu thresholding + adaptive threshold fusion. Divides the slice into upper/mid/lower regions by row percentile, applies closing on the upper region (fills holes in brain boundary) and opening on the lower region (removes vertical connections). Floodfill from (0,0) extracts the full brain mask. Ventricles = brain mask − processed tissue mask. Diagonal kernel morphological opening removes choroid plexus artifacts.

2. **`smooth_ventricles`** — Adaptive median blur that progressively reduces kernel size if the ventricle-to-brain ratio drops below 1% (prevents over-smoothing of small ventricles).

3. **`clean_ventricle_blobs`** — Connected component analysis that uses a "boat-shaped" ROI around the expected midline to prevent left/right frontal horn blobs from merging across the septum pellucidum. Returns the cleaned bilateral ventricle mask.

In [ ]:
def segment_coronal_plane(coronal_plane, params):
    """Segments ventricles and full brain mask from a 2D coronal MRI slice.

    MRI adaptation of the CT coronal pipeline: no HU windowing.
    Uses Otsu + adaptive threshold fusion with region-specific morphological ops.

    Parameters
    ----------
    coronal_plane : numpy.ndarray
        2D array of the coronal slice (shape: H x W).
    params : dict
        Segmentation parameters (coronal_seg_params).

    Returns
    -------
    ventricles : numpy.ndarray
        Binary mask of segmented ventricles.
    full_brain_mask : numpy.ndarray
        Binary mask of the full brain region.
    cX : int
        X-coordinate (column) of the brain mask centroid.
    cY : int
        Y-coordinate (row) of the brain mask centroid.
    """
    gauss_k = params["gauss_blur_kernel_size"]
    adapt_size = params["adaptive_thresh_size"]
    adapt_C = params["adaptive_thresh_C"]
    upper_pct = params["upper_percentile"]
    lower_pct = params["lower_percentile"]
    close_k = params["close_kernel_size"]
    close_iters = params["close_iterations"]
    open_k = params["open_kernel_size"]
    open_iters = params["open_iterations"]

    h, w = coronal_plane.shape
    empty_mask = np.zeros((h, w), dtype=np.uint8)

    # ── Gaussian blur ────────────────────────────────────────────────────────
    if gauss_k % 2 == 0:
        gauss_k += 1
    blurred = cv.GaussianBlur(normalize_to_uint8(coronal_plane), (gauss_k, gauss_k), cv.BORDER_DEFAULT)

    if blurred.max() == 0:
        return empty_mask.copy(), empty_mask.copy(), w // 2, h // 2

    # ── Otsu global threshold → binary ──────────────────────────────────────
    otsu_thresh = filters.threshold_otsu(blurred)
    otsu_bin = (blurred > otsu_thresh).astype(np.uint8)

    [cr, cc] = otsu_bin.nonzero()
    if len(cr) == 0:
        return empty_mask.copy(), empty_mask.copy(), w // 2, h // 2

    # ── Divide into upper / mid / lower by row percentile ───────────────────
    row_upper_boundary = int(np.percentile(cr, upper_pct))
    row_lower_boundary = int(np.percentile(cr, lower_pct))

    upper_mask = np.zeros((h, w), dtype=np.uint8)
    upper_mask[:row_upper_boundary, :] = 1
    mid_mask = np.zeros((h, w), dtype=np.uint8)
    mid_mask[row_upper_boundary:row_lower_boundary, :] = 1
    lower_mask = np.zeros((h, w), dtype=np.uint8)
    lower_mask[row_lower_boundary:, :] = 1

    # ── Morphological closing on upper region (fill holes in brain boundary) ─
    close_kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (close_k, close_k))
    upper_closed = cv.morphologyEx(
        np.uint8(otsu_bin * upper_mask), cv.MORPH_CLOSE, close_kernel, iterations=close_iters
    )
    # Merge: closed upper + original elsewhere
    combined = np.logical_or(upper_closed, otsu_bin * (1 - upper_mask)).astype(np.uint8)

    # ── Morphological opening on lower region (remove vertical connections) ──
    lower_open_kernel = np.array([[1, 1, 1], [0, 1, 0], [1, 1, 1]], dtype=np.uint8)
    lower_opened = cv.morphologyEx(
        np.uint8(combined * lower_mask), cv.MORPH_OPEN, lower_open_kernel, iterations=open_iters
    )
    combined = np.logical_or(combined * (1 - lower_mask), lower_opened).astype(np.uint8)

    # ── Full brain mask via floodfill from (0,0) ────────────────────────────
    im_ff = combined.copy().astype(np.uint8)
    ff_mask = np.zeros((h + 2, w + 2), np.uint8)
    res = cv.floodFill(im_ff, ff_mask, (0, 0), 255)
    full_brain_mask = (1 - res[2]).copy()[1:h + 1, 1:w + 1].astype(np.uint8)

    # Keep only the largest connected component
    labeled = measure.label(full_brain_mask, background=0)
    if labeled.max() == 0:
        return empty_mask.copy(), empty_mask.copy(), w // 2, h // 2
    full_brain_mask = getLargestCC(labeled).astype(np.uint8)

    # Centroid of full brain mask
    M = cv.moments(full_brain_mask)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = w // 2, h // 2

    # ── Adaptive threshold with background suppression ───────────────────────
    if adapt_size % 2 == 0:
        adapt_size += 1
    adapt_bin = cv.adaptiveThreshold(
        blurred, 1, cv.ADAPTIVE_THRESH_GAUSSIAN_C, cv.THRESH_BINARY,
        adapt_size, adapt_C
    )
    # Suppress background (outside brain)
    bg_rows, bg_cols = (1 - full_brain_mask).nonzero()
    adapt_suppressed = adapt_bin.copy()
    adapt_suppressed[bg_rows, bg_cols] = 0

    # ── Combine Otsu (mid) + adaptive (elsewhere) → global_local_mask ───────
    global_local_mask = np.logical_or(
        otsu_bin * mid_mask,
        adapt_suppressed * (1 - mid_mask)
    ).astype(np.uint8)

    # ── Ventricles = brain_mask - processed_tissue_mask ──────────────────────
    ventricles = np.clip(full_brain_mask.astype(np.int16) - global_local_mask.astype(np.int16), 0, 1).astype(np.uint8)

    # ── Remove choroid plexus with diagonal-kernel opening ───────────────────
    diag_kernel = np.array([[1, 0, 1], [0, 1, 0], [1, 0, 1]], dtype=np.uint8)
    ventricles = cv.morphologyEx(ventricles, cv.MORPH_OPEN, diag_kernel, iterations=1)

    return ventricles, full_brain_mask, cX, cY


def smooth_ventricles(ventricles, full_brain_mask, smoothing_factor=7):
    """Applies adaptive median blur to smooth ventricle boundaries.

    Reduces kernel size if the ventricle-to-brain ratio drops below 1%
    to prevent over-smoothing of small ventricles.

    Parameters
    ----------
    ventricles : numpy.ndarray
        2D binary mask of the segmented ventricles.
    full_brain_mask : numpy.ndarray
        2D binary mask of the full brain.
    smoothing_factor : int, optional
        Initial blur kernel size (must be odd; forced odd if even). Default is 7.

    Returns
    -------
    ventricles_smoothed : numpy.ndarray
        2D binary mask of the smoothed ventricles.
    ventricle_brain_ratio : float
        Ratio of smoothed ventricle area to total brain area.
    """
    total_brain_area = np.sum(full_brain_mask)
    if total_brain_area == 0:
        return np.zeros_like(ventricles), 0.0

    vent_8u = np.uint8(ventricles)
    k = smoothing_factor if smoothing_factor % 2 == 1 else smoothing_factor + 1

    ventricles_smoothed = cv.medianBlur(vent_8u, k)
    ratio = np.sum(ventricles_smoothed) / total_brain_area

    # Iteratively reduce kernel if ratio drops below 1%
    while ratio < 0.01 and k > 1:
        k = max(1, k - 2)
        ventricles_smoothed = cv.medianBlur(vent_8u, k)
        ratio = np.sum(ventricles_smoothed) / total_brain_area

    return ventricles_smoothed, ratio


def clean_ventricle_blobs(ventricles, full_brain_mask, cX, cY):
    """Cleans ventricle mask using connected component analysis and boat-shaped ROI.

    Creates a "boat-shaped" (V-on-rectangle) ROI around the midline to prevent
    left and right frontal horn blobs from merging across the septum pellucidum.
    After ROI masking, retains only the largest connected component per hemisphere.

    Parameters
    ----------
    ventricles : numpy.ndarray
        2D binary mask of the ventricles before cleaning.
    full_brain_mask : numpy.ndarray
        2D binary mask of the full brain.
    cX : int
        X-coordinate (column) of the brain centroid (used as midline estimate).
    cY : int
        Y-coordinate (row) of the brain centroid.

    Returns
    -------
    cleaned : numpy.ndarray
        2D binary mask with spurious blobs removed.
    """
    if np.sum(ventricles) == 0:
        return np.zeros_like(ventricles)

    h, w = ventricles.shape

    # ── Build boat-shaped ROI around midline ─────────────────────────────────
    # Get ventricle extent to size the ROI
    vr, vc = ventricles.nonzero()
    if len(vc) == 0:
        return np.zeros_like(ventricles)

    half_col_range = int((np.percentile(vc, 80) - np.percentile(vc, 20)) // 2)
    vcX = int(np.median(vc))

    boat_roi = np.zeros((h, w), dtype=np.uint8)
    r_start = max(0, cY - 5)
    r_end = min(h, cY + 15)
    c_left_start = max(0, cX - half_col_range)

    # Left side of the boat
    boat_roi[r_start:r_end, c_left_start:cX] = 1
    # Notch the upper-left corner to create the "V" shape
    m_thresh = (cY - 5) - (cX - half_col_range)
    for rm in range(max(0, cY - 5), min(h, cY + 5)):
        for cm in range(max(0, cX - half_col_range), min(w, cX)):
            if cm > (rm - m_thresh):
                boat_roi[rm, cm] = 0

    # Mirror right side
    left_width = cX - c_left_start
    right_end = min(w, cX + left_width)
    actual_w = right_end - cX
    if actual_w > 0:
        boat_roi[:, cX:right_end] = boat_roi[:, cX - actual_w:cX][:, ::-1]

    # ── Suppress blobs within the boat (midline suppression) ─────────────────
    # Keep ventricle pixels that are NOT inside the boat ROI, then union back
    # the original to prevent over-suppression on large ventricles
    outside_boat = np.logical_and(ventricles, 1 - boat_roi).astype(np.uint8)

    # Merge: start from outside-boat pixels, add boat pixels only if they
    # belong to a connected component that also has outside-boat support
    labeled_full = measure.label(ventricles, background=0)
    labeled_outside = measure.label(outside_boat, background=0)

    cleaned = np.zeros((h, w), dtype=np.uint8)

    if labeled_outside.max() == 0:
        # Nothing outside boat — fall back to full ventricle mask
        if labeled_full.max() > 0:
            try:
                cleaned = getLargestCC(labeled_full).astype(np.uint8)
            except ValueError:
                pass
        return cleaned

    # For each connected component in outside_boat, keep the corresponding
    # pixels in the full ventricle mask (including any boat-region parts of the same blob)
    outside_labels = np.unique(labeled_outside[labeled_outside > 0])
    for lbl in outside_labels:
        outside_rows, outside_cols = np.where(labeled_outside == lbl)
        # Find which full-mask labels overlap
        overlapping_full_labels = np.unique(labeled_full[outside_rows, outside_cols])
        for fl in overlapping_full_labels:
            if fl > 0:
                cleaned[labeled_full == fl] = 1

    # ── Final: keep only the two largest blobs (left + right frontal horn) ───
    labeled_cleaned = measure.label(cleaned, background=0)
    if labeled_cleaned.max() == 0:
        return np.zeros_like(ventricles)

    counts = np.bincount(labeled_cleaned.flatten())
    counts[0] = 0  # ignore background
    n_keep = min(2, labeled_cleaned.max())
    top_labels = np.argsort(counts)[::-1][:n_keep]

    result = np.zeros((h, w), dtype=np.uint8)
    for lbl in top_labels:
        result[labeled_cleaned == lbl] = 1

    return result

## Core function: calc_ei_z()

In [ ]:
def calc_ei_z(img_arr, ac_indices, params, visualize=False):
    """Calculates Evans Index Z from coronal MRI slices.

    Iterates coronal slices from AC downward (toward inferior), segments ventricles
    on each slice, measures the maximum frontal horn height on both left and right
    sides, and normalizes by the brain height at the optimal slice.

    For OASIS atlas-registered isotropic volumes, coronal slices are extracted via
    direct numpy indexing: ``arr[:, y, :]``.
    For ABCD, pass the array from ``extract_coronal_view()`` instead.

    Parameters
    ----------
    img_arr : numpy.ndarray
        3D array of the MRI volume, shape (Z, Y, X) after rotate_image().
    ac_indices : tuple or list
        Voxel-space AC coordinates (x_vox, y_vox, z_vox). The y_vox index
        gives the starting coronal slice; the z_vox index gives the axial
        position used as the inferior AC boundary within each coronal slice.
    params : dict
        Segmentation parameters dict (coronal_seg_params).
    visualize : bool, optional
        If True, plots the optimal coronal slice with measurement overlays.
        Default is False.

    Returns
    -------
    ei_z : float or None
        The computed Evans Index Z value, or None if computation failed.
    opt_plane : int or None
        The coronal slice index (Y-axis) at which EI-z was maximized,
        or None if computation failed.
    """
    num_slices_below_ac = params["num_slices_below_ac"]

    ac_x, ac_y, ac_z = int(round(ac_indices[0])), int(round(ac_indices[1])), int(round(ac_indices[2]))

    # In coronal slices arr[:, y, :], the Z-axis (array dim 0) is S→I.
    # ac_z gives the voxel row corresponding to the AC in each coronal slice.
    # We measure frontal horn height from the top of the horn down to that row.
    ac_row_in_coronal = int(img_arr.shape[0] - ac_z)  # flip because array dim 0 is S→I

    max_fhh = 0
    opt_plane = None
    opt_full_brain_mask = None
    opt_coronal_plane = None
    opt_ventricles_final = None
    opt_left_flag = 0
    opt_fh_x = None
    opt_fh_y1 = None
    opt_fh_y2 = None
    opt_sorted_unique_cols = None

    # Iterate from AC coronal slice downward (inferior direction = decreasing Y in OASIS atlas)
    y_start = ac_y
    y_end = max(0, ac_y - num_slices_below_ac)

    for y in range(y_start, y_end - 1, -1):
        # Extract coronal slice: arr[:, y, :] gives (Z, X) array
        coronal_plane = img_arr[:, y, :]

        if coronal_plane.max() == 0:
            continue

        # Normalize to uint8 for OpenCV processing
        plane_min, plane_max = coronal_plane.min(), coronal_plane.max()
        if plane_max == plane_min:
            continue
        plane_u8 = np.uint8(255.0 * (coronal_plane - plane_min) / (plane_max - plane_min))

        # Segment
        ventricles, full_brain_mask, cX, cY = segment_coronal_plane(plane_u8, params)

        if np.sum(full_brain_mask) == 0:
            continue

        # Clean ventricle blobs (boat-shaped midline suppression)
        cleaned = clean_ventricle_blobs(ventricles, full_brain_mask, cX, cY)

        if np.sum(cleaned) == 0:
            continue

        # Smooth
        smoothed, vent_brain_ratio = smooth_ventricles(cleaned, full_brain_mask)

        if vent_brain_ratio < 0.01:
            continue

        # Restrict to frontal horn region: above AC row in coronal slice
        above_ac_mask = np.zeros(coronal_plane.shape, dtype=np.uint8)
        above_ac_mask[:ac_row_in_coronal, :] = 1
        smoothed_frontal = np.where(smoothed * above_ac_mask > 0, 1, 0).astype(np.uint8)

        if np.sum(smoothed_frontal) == 0:
            continue

        vr, vc = smoothed_frontal.nonzero()
        if len(vr) == 0:
            continue

        # Skip if ventricle blob is taller than 1.5x its width (spurious vertical streak)
        vent_h = max(vr) - min(vr)
        vent_w = max(vc) - min(vc)
        if vent_w > 0 and vent_h > 1.5 * vent_w:
            continue

        # ── Measure frontal horn heights ──────────────────────────────────────
        # Split left / right at midline column
        mvc = (max(vc) + min(vc)) // 2  # midline column of ventricle bounding box
        sorted_cols = sorted(vc)
        sorted_cols_left = [c for c in sorted_cols if c <= mvc - 2]
        sorted_cols_right = [c for c in sorted_cols if c > mvc + 2]

        if len(sorted_cols_left) == 0 or len(sorted_cols_right) == 0:
            continue

        # Top row for each column (highest point in frontal horn)
        top_rows_left = [min(vr[vc == c]) for c in sorted_cols_left if np.sum(vc == c) > 0]
        top_rows_right = [min(vr[vc == c]) for c in sorted_cols_right if np.sum(vc == c) > 0]

        if len(top_rows_left) == 0 or len(top_rows_right) == 0:
            continue

        # Frontal horn height = distance from top of horn to AC row
        opt_ly1 = min(top_rows_left)
        opt_ry1 = min(top_rows_right)
        max_l_dist = ac_row_in_coronal - opt_ly1
        max_r_dist = ac_row_in_coronal - opt_ry1

        if max_l_dist <= 0 and max_r_dist <= 0:
            continue

        fhh = max(max_l_dist, max_r_dist)
        left_flag = 1 if max_l_dist >= max_r_dist else 0

        opt_lx = min(np.array(sorted_cols_left)[np.array(top_rows_left) == opt_ly1])
        opt_rx = max(np.array(sorted_cols_right)[np.array(top_rows_right) == opt_ry1])

        if fhh > max_fhh:
            max_fhh = fhh
            opt_plane = y
            opt_left_flag = left_flag
            opt_full_brain_mask = full_brain_mask
            opt_coronal_plane = plane_u8
            opt_ventricles_final = smoothed_frontal
            opt_fh_x_left = opt_lx
            opt_fh_x_right = opt_rx
            opt_fh_y1_left = opt_ly1
            opt_fh_y1_right = opt_ry1
            opt_fh_y2 = ac_row_in_coronal
            opt_sorted_unique_cols = np.unique(sorted_cols)

    if max_fhh == 0 or opt_full_brain_mask is None:
        return None, None

    # ── Brain height measurement at optimal slice ─────────────────────────────
    # Brain height = maximum vertical extent of the brain mask excluding the
    # columns occupied by the ventricles (following CT EI-z convention)
    br, bc = (opt_full_brain_mask > 0).nonzero()
    if len(br) == 0:
        return None, None

    vent_col_min = int(np.min(opt_sorted_unique_cols))
    vent_col_max = int(np.max(opt_sorted_unique_cols))

    # Use brain columns outside the ventricle column range
    brain_cols_outside = [c for c in np.unique(bc) if c < vent_col_min or c > vent_col_max]
    if len(brain_cols_outside) == 0:
        brain_cols_outside = list(np.unique(bc))  # fall back to all columns

    max_brain_height = 0
    opt_bx = brain_cols_outside[0]
    opt_by1, opt_by2 = 0, 0

    for col in brain_cols_outside:
        col_rows = br[bc == col]
        if len(col_rows) < 2:
            continue
        bh = max(col_rows) - min(col_rows)
        if bh > max_brain_height:
            max_brain_height = bh
            opt_bx = col
            opt_by1 = min(col_rows)
            opt_by2 = max(col_rows)

    if max_brain_height == 0:
        return None, None

    ei_z = max_fhh / max_brain_height

    if visualize:
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(opt_coronal_plane, cmap='gray')
        ax.set_title(f"Optimal coronal slice y={opt_plane}  |  EI-z = {ei_z:.3f}")

        # Draw winning frontal horn measurement
        if opt_left_flag:
            fh_x, fh_y1 = opt_fh_x_left, opt_fh_y1_left
        else:
            fh_x, fh_y1 = opt_fh_x_right, opt_fh_y1_right
        ax.scatter([fh_x] * (opt_fh_y2 - fh_y1 + 1),
                   list(range(fh_y1, opt_fh_y2 + 1)),
                   marker='x', s=4, color='cyan', label='Frontal horn height')

        # Draw brain height measurement
        ax.scatter([opt_bx] * (opt_by2 - opt_by1 + 1),
                   list(range(opt_by1, opt_by2 + 1)),
                   marker='x', s=4, color='yellow', label='Brain height')

        ax.legend()
        plt.tight_layout()
        plt.show()

    return ei_z, opt_plane

## Load demographics (optional)

If a demographics file is provided, load it for later merging with the computed EI-z values.

In [ ]:
demo_df = None
if DEMOGRAPHICS_PATH and os.path.exists(DEMOGRAPHICS_PATH):
    ext = os.path.splitext(DEMOGRAPHICS_PATH)[1].lower()
    if ext in (".xlsx", ".xls"):
        demo_df = pd.read_excel(DEMOGRAPHICS_PATH)
    elif ext == ".csv":
        demo_df = pd.read_csv(DEMOGRAPHICS_PATH)
    elif ext in (".tsv", ".txt"):
        demo_df = pd.read_csv(DEMOGRAPHICS_PATH, sep="\t")
    else:
        print(f"Unknown demographics format: {ext}")

    if demo_df is not None:
        print(f"Loaded demographics: {len(demo_df)} subjects, columns: {list(demo_df.columns)}")
        demo_df.head()
else:
    print("No demographics file — skipping clinical metadata.")

## Discover available scans

Uses the `SCAN_GLOB` pattern from the configuration cell to find all volumes in `DATA_DIR`.

In [ ]:
scan_files = sorted(glob.glob(os.path.join(DATA_DIR, SCAN_GLOB), recursive=True))

scan_df = pd.DataFrame({
    "subject_id": [SUBJECT_ID_FROM_PATH(p) for p in scan_files],
    "scan_path": scan_files,
})
print(f"Found {len(scan_df)} scans in {DATA_DIR}")
scan_df.head()

## Load AC coordinates

For **OASIS** atlas-registered volumes the AC is a fixed atlas voxel (`OASIS_AC_VOXEL` from `mri_utils`).
For **ABCD**, AC/PC are loaded per-subject from `.txt` sidecars using `load_acpc_from_txt()`.

In [ ]:
def get_ac_indices(subject_id, scan_path, ac_mode, acpc_txt_glob=None):
    """Returns AC voxel coordinates (x, y, z) for a subject.

    Parameters
    ----------
    subject_id : str
        Subject identifier (used to find sidecar files).
    scan_path : str
        Path to the NIfTI volume.
    ac_mode : str
        One of 'oasis_atlas' (use OASIS_AC_VOXEL) or 'txt_sidecar'
        (load per-subject from a .txt file).
    acpc_txt_glob : str or None
        Glob pattern for finding ACPC .txt sidecars (used with 'txt_sidecar').

    Returns
    -------
    ac : tuple of int or None
        (x_vox, y_vox, z_vox), or None if not found.
    """
    if ac_mode == "oasis_atlas":
        return OASIS_AC_VOXEL  # (85, 110, 73) — fixed for T88 atlas

    elif ac_mode == "txt_sidecar":
        if acpc_txt_glob is None:
            raise ValueError("acpc_txt_glob must be provided for txt_sidecar mode.")
        scan_dir = os.path.dirname(scan_path)
        txt_files = glob.glob(os.path.join(scan_dir, acpc_txt_glob), recursive=True)
        if not txt_files:
            # Also try DATA_DIR-relative pattern
            txt_files = glob.glob(os.path.join(DATA_DIR, f"**/{subject_id}*acpc*.txt"), recursive=True)
        if not txt_files:
            print(f"  No ACPC .txt found for {subject_id}")
            return None
        ac, pc = load_acpc_from_txt(txt_files[0])
        # Convert physical AC coordinates to voxel indices using the loaded image
        img_tmp = sitk.ReadImage(scan_path)
        img_tmp_rot = rotate_image(img_tmp)
        ac_vox = img_tmp_rot.TransformPhysicalPointToContinuousIndex(ac)
        return (int(round(ac_vox[0])), int(round(ac_vox[1])), int(round(ac_vox[2])))

    else:
        raise ValueError(f"Unknown ac_mode: {ac_mode!r}")


# Quick test: print AC indices for the first subject
if len(scan_df) > 0:
    test_ac = get_ac_indices(
        scan_df.iloc[0]["subject_id"],
        scan_df.iloc[0]["scan_path"],
        AC_MODE,
        ACPC_TXT_GLOB
    )
    print(f"AC indices for {scan_df.iloc[0]['subject_id']}: {test_ac}")

## Test on a single subject first

Run the full EI-z pipeline on one scan to verify the segmentation before batch processing.

**What to expect:**
- The optimal coronal slice should show two symmetric dark CSF cavities (frontal horns) with cyan measurement markers running vertically through each horn.
- Yellow markers show the brain height column used for normalization.
- A healthy young adult EI-z is typically 0.20–0.30; hydrocephalus is often > 0.40.

In [ ]:
test_row = scan_df.iloc[0]
test_id = test_row["subject_id"]
test_path = test_row["scan_path"]

print(f"Testing on: {test_id}")
print(f"Path: {test_path}")

# Load and orient
test_img = sitk.ReadImage(test_path)
test_img_rot = rotate_image(test_img)
test_arr = sitk.GetArrayFromImage(test_img_rot)

print(f"Array shape (Z, Y, X): {test_arr.shape}")
print(f"Spacing (X, Y, Z): {test_img_rot.GetSpacing()}")

# Get AC indices
test_ac = get_ac_indices(test_id, test_path, AC_MODE, ACPC_TXT_GLOB)
print(f"AC voxel (x, y, z): {test_ac}")

# Show a representative coronal slice near AC
y_preview = test_ac[1]
preview_plane = test_arr[:, y_preview, :]
preview_min, preview_max = preview_plane.min(), preview_plane.max()
if preview_max > preview_min:
    preview_u8 = np.uint8(255.0 * (preview_plane - preview_min) / (preview_max - preview_min))
else:
    preview_u8 = np.zeros_like(preview_plane, dtype=np.uint8)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(preview_u8, cmap="gray")
ax.set_title(f"{test_id} — Coronal slice at AC (y={y_preview})")
ax.axhline(y=test_arr.shape[0] - test_ac[2], color='red', linestyle='--', linewidth=1, label='AC row (inferior boundary)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Run EI-z with visualization on the test subject
ei_z_test, opt_plane_test = calc_ei_z(test_arr, test_ac, coronal_seg_params, visualize=True)

if ei_z_test is not None:
    print(f"\nEI-z for {test_id}: {ei_z_test:.4f}  (optimal coronal slice y={opt_plane_test})")
else:
    print(f"\nCould not compute EI-z for {test_id}. Check segmentation outputs above.")

## Batch processing — all subjects

Run the EI-z pipeline on all discovered scans with visualization off.

In [ ]:
results = []

for _, row in scan_df.iterrows():
    subject_id = row["subject_id"]
    vol_path = row["scan_path"]

    try:
        # Load and orient
        img = sitk.ReadImage(vol_path)
        img_rot = rotate_image(img)
        img_arr = sitk.GetArrayFromImage(img_rot)

        # Get AC indices
        ac_indices = get_ac_indices(subject_id, vol_path, AC_MODE, ACPC_TXT_GLOB)
        if ac_indices is None:
            print(f"{subject_id}: SKIP — no AC coordinates")
            results.append({"subject_id": subject_id, "ei_z": np.nan, "opt_coronal_plane": np.nan})
            continue

        # Compute EI-z
        ei_z, opt_plane = calc_ei_z(img_arr, ac_indices, coronal_seg_params, visualize=False)

        results.append({
            "subject_id": subject_id,
            "ei_z": ei_z,
            "opt_coronal_plane": opt_plane,
        })

        if ei_z is not None:
            print(f"{subject_id}: EI-z={ei_z:.4f}  (y={opt_plane})")
        else:
            print(f"{subject_id}: FAILED — could not compute EI-z")

    except Exception as e:
        print(f"{subject_id}: ERROR — {e}")
        results.append({"subject_id": subject_id, "ei_z": np.nan, "opt_coronal_plane": np.nan})

results_df = pd.DataFrame(results)
n_ok = results_df["ei_z"].notna().sum()
print(f"\nProcessed {len(results_df)} subjects  |  success: {n_ok}  |  failed: {len(results_df) - n_ok}")

## Results analysis

**What to expect:**

1. **EI-z distribution (left):** Most healthy adults fall in the range 0.20–0.35. Values > 0.40 suggest ventriculomegaly (consistent with the clinical threshold for Evans Index).

2. **EI-z by CDR group (right, OASIS only):** CDR 0 (healthy) subjects should cluster at lower values; CDR 0.5/1.0 (dementia) subjects tend to have larger ventricles and should shift rightward.

In [ ]:
# Merge with demographics if available
if demo_df is not None:
    id_col = next(
        (c for c in demo_df.columns if c.upper() in ("ID", "SUBJECT_ID", "SUBJECTKEY", "SRC_SUBJECT_ID")),
        None
    )
    if id_col:
        merge_cols = [id_col] + [
            c for c in demo_df.columns
            if c != id_col and c in ("Age", "M/F", "CDR", "Sex", "age", "sex", "interview_age")
        ]
        results_df = results_df.merge(
            demo_df[merge_cols], left_on="subject_id", right_on=id_col, how="left"
        )
        print(f"Merged demographics on column: {id_col}")

has_cdr = "CDR" in results_df.columns and results_df["CDR"].notna().any()
n_plots = 1 + int(has_cdr)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

# EI-z distribution
valid_eiz = results_df["ei_z"].dropna()
axes[0].hist(valid_eiz, bins=20, edgecolor="black")
axes[0].axvline(x=0.40, color="red", linestyle="--", label="Ventriculomegaly threshold (0.40)")
axes[0].set_xlabel("EI-z")
axes[0].set_ylabel("Count")
axes[0].set_title(f"EI-z Distribution (n={len(valid_eiz)})")
axes[0].legend()

# EI-z by CDR
if has_cdr:
    cdr_colors = {0.0: "steelblue", 0.5: "darkorange", 1.0: "crimson"}
    for cdr_val in sorted(results_df["CDR"].dropna().unique()):
        subset = results_df[results_df["CDR"] == cdr_val]["ei_z"].dropna()
        if len(subset) > 0:
            color = cdr_colors.get(cdr_val, "gray")
            axes[1].hist(subset, bins=15, alpha=0.55, label=f"CDR {cdr_val} (n={len(subset)})",
                         color=color, edgecolor="black")
    axes[1].axvline(x=0.40, color="red", linestyle="--", label="Threshold 0.40")
    axes[1].set_xlabel("EI-z")
    axes[1].set_ylabel("Count")
    axes[1].set_title("EI-z by Dementia Severity (CDR)")
    axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nEI-z summary (n={len(valid_eiz)}):")
print(valid_eiz.describe().round(4))

if has_cdr:
    print("\nEI-z by CDR group:")
    print(results_df.groupby("CDR")["ei_z"].describe().round(4))

## Quality check — extreme values

Inspect subjects with EI-z outside the expected physiological range (< 0.10 or > 0.60) for potential segmentation errors.

In [ ]:
min_ei_z_lim = 0.10
max_ei_z_lim = 0.60

outliers = results_df[(results_df["ei_z"] <= min_ei_z_lim) | (results_df["ei_z"] >= max_ei_z_lim)]
print(f"Outliers (EI-z ≤ {min_ei_z_lim} or ≥ {max_ei_z_lim}): {len(outliers)} subjects")
outliers

## Save results to CSV

In [ ]:
output_path = os.path.join(OUTPUT_DIR, f"ei_z_{DATASET}.csv")
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
print(f"Shape: {results_df.shape}")
results_df.head()